# SwinIR SEM 微调 — 正式训练

GPU: 2×T4 | Checkpoint: 10k iter | 目标: 70k iter

**策略**：
1. 先跑 200 iter 试运行（验证训练能启动、ETA 合理）
2. 确认无误后，跑完整训练
3. 训练完成后下载 output 中的模型和日志

## 0. 环境准备（与 check.ipynb 相同）

In [1]:
import torch
NUM_GPUS = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0))
    print(f'  GPU {i}: {p.name} — {mem/1e9:.1f} GB')

# 按 GPU 数量和速度预估安全的 total_iter
# P100 ≈ 1.0s/iter, T4 ≈ 1.2s/iter, 取保守值 1.2s
SEC_PER_ITER = 1.2
MAX_SECONDS = 11.5 * 3600  # 留 30min 余量
MAX_NEW_ITERS = int(MAX_SECONDS / SEC_PER_ITER / NUM_GPUS) if NUM_GPUS > 0 else 35000
TOTAL_ITER = 10000 + MAX_NEW_ITERS
print(f'\n预估: {NUM_GPUS}×GPU, ~{SEC_PER_ITER}s/iter → total_iter={TOTAL_ITER} (安全值)')

PyTorch: 2.10.0+cu128, CUDA: True, GPUs: 1
  GPU 0: Tesla P100-PCIE-16GB — 17.1 GB

预估: 1×GPU, ~1.2s/iter → total_iter=44500 (安全值)


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [2]:
import os, shutil
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

# 克隆仓库
if not os.path.exists('BasicSR'):
    !git clone https://github.com/Log-Dog012/BasicSR.git
    !cd BasicSR && git checkout cuda-sem-finetune

# 安装依赖
os.chdir('BasicSR')
!pip install -r requirements.txt -q
!pip install -e . -q
!pip install lpips timm -q
print('✅ 环境就绪')

Cloning into 'BasicSR'...
remote: Enumerating objects: 4830, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 4830 (delta 84), reused 88 (delta 56), pack-reused 4689 (from 3)
Receiving objects: 100% (4830/4830), 4.07 MiB | 30.00 MiB/s, done.
Resolving deltas: 100% (3200/3200), done.
Already on 'cuda-sem-finetune'
Your branch is up to date with 'origin/cuda-sem-finetune'.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.1 MB/s eta 0:00:00
✅ 环境就绪


In [3]:
# 复制模型文件
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/1'
swinir_zoo = os.path.join(WORK_DIR, 'BasicSR', 'SwinIR', 'model_zoo')
exp_dir = os.path.join(WORK_DIR, 'BasicSR', 'experiments', 'finetune_SwinIR_SRx4_SEM')

# 预训练权重
os.makedirs(swinir_zoo, exist_ok=True)
for root, dirs, files in os.walk(MODEL_INPUT):
    for f in files:
        if 'classicalSR' in f and f.endswith('.pth'):
            shutil.copy2(os.path.join(root, f), os.path.join(swinir_zoo, f))
            print(f'✅ 预训练: {f}')
            break

# Checkpoint
for name in ['net_g_10000.pth', '10000.state']:
    for root, dirs, files in os.walk(MODEL_INPUT):
        if name in files:
            dst_dir = os.path.join(exp_dir, 'models' if 'pth' in name else 'training_states')
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(os.path.join(root, name), os.path.join(dst_dir, name))
            print(f'✅ {name}')
            break
print('✅ 模型文件就绪')

✅ 预训练: 001_classicalSR_DIV2K_s48w8_SwinIR-M_x4.pth
✅ net_g_10000.pth
✅ 10000.state
✅ 模型文件就绪


## 1. 正式训练

根据 GPU 数量自动计算安全的 `total_iter`（留 30min 余量防 12h 超时）。
- 2×T4: ~39000 iter → total_iter=49000
- 1×P100: ~32000 iter → total_iter=42000
- `auto_resume` 确保超时后下次运行自动恢复

In [4]:
# 正式训练 — 自适应 GPU 数量
# --force_yml 覆盖 total_iter 为安全值，不改原 YAML
import subprocess, sys

print(f'开始正式训练 ({TOTAL_ITER} iter, {NUM_GPUS}×GPU)...')
print('='*50)

if NUM_GPUS >= 2:
    # DDP 多卡
    !torchrun --nproc_per_node={NUM_GPUS} --master_port=4321 -m basicsr.train \
      -opt options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml \
      --launcher pytorch --auto_resume \
      --force_yml train:total_iter={TOTAL_ITER}
else:
    # 单卡，不用 torchrun
    !python -m basicsr.train \
      -opt options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml \
      --auto_resume \
      --force_yml train:total_iter={TOTAL_ITER}

开始正式训练 (44500 iter, 1×GPU)...
<frozen runpy>:128: RuntimeWarning: 'basicsr.train' found in sys.modules after import of package 'basicsr', but prior to execution of 'basicsr.train'; this may result in unpredictable behaviour
Disable distributed.
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabi

## 3. 结果分析

In [5]:
# 提取所有验证结果
logs = sorted([f for f in os.listdir(exp_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(exp_dir, logs[-1])
    with open(latest_log, 'r') as f:
        content = f.read()
    
    # 提取验证 PSNR
    import re
    val_matches = re.findall(r'psnr:\s+([\d.]+)\s+Best:\s+([\d.]+)\s+@\s+(\d+)', content)
    
    if val_matches:
        print('验证结果:')
        print(f'{"Iter":<10} {"PSNR":<10} {"Best PSNR":<12}')
        print('-' * 35)
        best_psnr = 0
        for psnr_val, best_val, iter_val in val_matches:
            marker = ' ←' if float(psnr_val) == float(best_val) else ''
            print(f'{iter_val:<10} {psnr_val:<10} {best_val:<12}{marker}')
    
    # 最后几行
    lines = content.strip().split('\n')
    print(f'\n日志最后 5 行:')
    for line in lines[-5:]:
        print(line)
else:
    print('未找到日志')


日志最后 5 行:
2026-07-09 17:24:07,386 INFO: Loading SwinIR model from /kaggle/working/BasicSR/experiments/finetune_SwinIR_SRx4_SEM/models/net_g_10000.pth, with param key: [params_ema].
2026-07-09 17:24:07,463 INFO: Loss [L1Loss] is created.
2026-07-09 17:24:07,466 INFO: Model [SwinIRModel] is created.
2026-07-09 17:24:07,474 INFO: Resuming training from epoch: 2, iter: 10000.
2026-07-09 17:24:07,611 INFO: Start training from epoch: 2, iter: 10000


## 4. 保存结果到 output

训练完成后，output 目录中的文件会被保存为 Kaggle Dataset，可以下载。

In [6]:
# /kaggle/working 整个目录在 Save and Run All 后会自动保存为 output
# 不需要手动复制，直接查看训练产物即可

print('训练产物位置:')
print(f'  模型权重: {models_dir}')
print(f'  训练状态: {states_dir}')

# 列出模型文件
print(f'\n模型文件:')
for f in sorted(os.listdir(models_dir)):
    size = os.path.getsize(os.path.join(models_dir, f)) / 1e6
    print(f'  {f} ({size:.1f}MB)')

# 列出训练状态文件
print(f'\n训练状态:')
for f in sorted(os.listdir(states_dir)):
    size = os.path.getsize(os.path.join(states_dir, f)) / 1e6
    print(f'  {f} ({size:.1f}MB)')

# 列出日志
print(f'\n日志文件:')
for f in sorted(os.listdir(exp_dir)):
    if f.endswith('.log'):
        size = os.path.getsize(os.path.join(exp_dir, f)) / 1e6
        print(f'  {f} ({size:.1f}MB)')


训练产物位置:


NameError: name 'models_dir' is not defined

In [ ]:
# Kaggle Save and Run All 会自动保存 /kaggle/working 下所有内容
# 右侧面板 → Data → 下载即可

working_size = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk('/kaggle/working')
    for f in files
)
print(f'working 目录总大小: {working_size/1e9:.2f} GB')
print(f'训练完成！Save and Run All 后，右侧面板 → Data → 下载。')